# Figure 3 — Q2: with the truth in hand, what did the methods actually do?

Mock communities and simulations both carry a known answer. This figure asks what was removed, what was kept, and whether the proxy metrics of Figure 2 track any of it.

Each panel has its own cell and renders on its own, sized to suit it alone. The last cell assembles all of them at the submission size (180 x 235 mm) and writes the PDF.

The panel functions live in `figures/fig3.R` and are **sourced, not copied** — edit them there and re-run the cell, so the notebook and `snakemake` produce the same figure.

## Setup

In [ ]:
# Panels come from the script, so the notebook and the submitted figure cannot drift apart.
# main() is guarded, so sourcing runs nothing.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))
source("figures/fig3.R")

results_dir <- "results"

## Data

In [ ]:
scored <- read_scores(file.path(results_dir, "scores.tsv"))
simulated <- scored[scored$regime == "sim" & scored$status == "ok" & scored$method != "oracle", ]
truth_summary <- read.delim(file.path(results_dir, "truth_summary.tsv"))
truth_summary <- truth_summary[!is.na(truth_summary$aitchison_to_truth), ]
per_taxon <- read.delim(file.path(results_dir, "truth_taxon.tsv"))

cat(nrow(simulated), "simulated rows,", nrow(truth_summary), "known-composition rows\n")

## a. Per-taxon bias against the pipetted truth

Taxa ordered by their bias before correction. Genus names abbreviated and italicised; spelled out they cost a quarter of the figure's height.

In [ ]:
show_panel(panel_taxon_bias(per_taxon, "real/jmbc_extraction"), 180, 78)

## b. Accuracy against the pipetted truth

Aitchison distance to the known composition, per mock dataset. A missing bar is a method that returned no result there.

In [ ]:
show_panel(panel_distance_to_truth(truth_summary), 100, 78)

## c. False structure between protocols

Identical material, many protocols: whatever batch variance survives is spurious by construction.

In [ ]:
show_panel(panel_false_structure(truth_summary), 100, 78)

## d. Fidelity to the simulated truth

Agreement with the batch-free oracle table, as distance fidelity and as per-taxon fidelity.

In [ ]:
show_panel(panel_oracle_fidelity(simulated), 100, 82)

## e. Does the multiplicative assumption decide it?

Per-taxon fidelity split by how much of the community the bias touched.

In [ ]:
show_panel(panel_systematic(simulated), 100, 82)

## f. Proxy against truth

One point per scenario: batch variance removed against fidelity gained, both relative to raw. The lower-left quadrant is correction that removed batch and lost the truth with it.

In [ ]:
show_panel(panel_proxy_versus_truth(simulated), 170, 82)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 235 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it.

In [ ]:
figure <- (panel_taxon_bias(per_taxon, "real/jmbc_extraction") /
           (panel_distance_to_truth(truth_summary) | panel_false_structure(truth_summary)) /
           (panel_oracle_fidelity(simulated) | panel_systematic(simulated)) /
           panel_proxy_versus_truth(simulated)) +
  plot_layout(heights = c(1, 1.1, 1.1, 0.8)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 9, face = "bold"))

save_figure(figure, "results/figures/fig3.pdf", 180, 235)
show_panel(figure, 180, 235, scale = 1.5)